# **Importing Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
dataset = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
print(dataset.head(5))

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [ ]:
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [ ]:
print(X)

[['7590-VHVEG' 'Female' 0 ... 'Electronic check' 29.85 '29.85']
 ['5575-GNVDE' 'Male' 0 ... 'Mailed check' 56.95 '1889.5']
 ['3668-QPYBK' 'Male' 0 ... 'Mailed check' 53.85 '108.15']
 ...
 ['4801-JZAZL' 'Female' 0 ... 'Electronic check' 29.6 '346.45']
 ['8361-LTMKD' 'Male' 1 ... 'Mailed check' 74.4 '306.6']
 ['3186-AJIEK' 'Male' 0 ... 'Bank transfer (automatic)' 105.65 '6844.5']]


In [ ]:
print(y)

['No' 'No' 'Yes' ... 'No' 'Yes' 'No']


# **Encoding Dependant Variable**

In [ ]:
#Encoding Dependant variable
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
print(y)

[0 0 1 ... 0 1 0]


In [ ]:
summary = pd.DataFrame({
    "dtype": dataset.dtypes,
    "n_unique": dataset.nunique(),
    "missing": dataset.isna().sum()
}).sort_values("n_unique")
print(summary)

# For text columns with few unique values, list the actual values
for col in dataset.select_dtypes(exclude="number").columns:
    if dataset[col].nunique() <= 10:
        print(f"{col:18s} -> {dataset[col].unique()}")

                    dtype  n_unique  missing
gender             object         2        0
SeniorCitizen       int64         2        0
Partner            object         2        0
Dependents         object         2        0
PhoneService       object         2        0
PaperlessBilling   object         2        0
Churn              object         2        0
MultipleLines      object         3        0
TechSupport        object         3        0
StreamingTV        object         3        0
OnlineBackup       object         3        0
DeviceProtection   object         3        0
StreamingMovies    object         3        0
Contract           object         3        0
OnlineSecurity     object         3        0
InternetService    object         3        0
PaymentMethod      object         4        0
tenure              int64        73        0
MonthlyCharges    float64      1585        0
TotalCharges       object      6531        0
customerID         object      7043        0
gender    

In [ ]:
print(type(X))

<class 'pandas.core.frame.DataFrame'>


Dropping Customer ID

In [ ]:
X = dataset.iloc[:, 1:-1].values

In [ ]:
print(X)

[['Female' 0 'Yes' ... 'Electronic check' 29.85 '29.85']
 ['Male' 0 'No' ... 'Mailed check' 56.95 '1889.5']
 ['Male' 0 'No' ... 'Mailed check' 53.85 '108.15']
 ...
 ['Female' 0 'Yes' ... 'Electronic check' 29.6 '346.45']
 ['Male' 1 'Yes' ... 'Mailed check' 74.4 '306.6']
 ['Male' 0 'No' ... 'Bank transfer (automatic)' 105.65 '6844.5']]


# Encoding Categories

In [ ]:
# TotalCharges = index 18, tenure = index 4
tc = X[:, 18].astype(str)
blank = np.char.strip(tc) == ""

print("Blank TotalCharges:", blank.sum())
print("Tenure of those rows:", X[blank, 4])

Blank TotalCharges: 11
Tenure of those rows: [0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
tc[blank] = "0"
X[:, 18] = tc.astype(float)

In [ ]:
print("Still blank:", np.sum(np.char.strip(X[:, 18].astype(str)) == ""))   # expect 0
print(X[:5, 18])

Still blank: 0
[29.85 1889.5 108.15 1840.75 151.65]


In [ ]:
for i in [0, 2, 3, 5, 15]:        # gender, Partner, Dependents, PhoneService, PaperlessBilling
    X[:, i] = le.fit_transform(X[:, i])

In [ ]:
print(X)

[[0 0 1 ... 'Electronic check' 29.85 29.85]
 [1 0 0 ... 'Mailed check' 56.95 1889.5]
 [1 0 0 ... 'Mailed check' 53.85 108.15]
 ...
 [0 0 1 ... 'Electronic check' 29.6 346.45]
 [1 1 1 ... 'Mailed check' 74.4 306.6]
 [1 0 0 ... 'Bank transfer (automatic)' 105.65 6844.5]]


In [ ]:
names = dataset.columns[1:-1]   # column names, for reference only

for i in range(X.shape[1]):
    vals = np.unique(X[:, i].astype(str))
    preview = vals[:6] if len(vals) > 6 else vals
    print(f"{i:2d} {names[i]:18s} n_unique={len(vals):5d}  {preview}")


 0 gender             n_unique=    2  ['0' '1']
 1 SeniorCitizen      n_unique=    2  ['0' '1']
 2 Partner            n_unique=    2  ['0' '1']
 3 Dependents         n_unique=    2  ['0' '1']
 4 tenure             n_unique=   73  ['0' '1' '10' '11' '12' '13']
 5 PhoneService       n_unique=    2  ['0' '1']
 6 MultipleLines      n_unique=    3  ['No' 'No phone service' 'Yes']
 7 InternetService    n_unique=    3  ['DSL' 'Fiber optic' 'No']
 8 OnlineSecurity     n_unique=    3  ['No' 'No internet service' 'Yes']
 9 OnlineBackup       n_unique=    3  ['No' 'No internet service' 'Yes']
10 DeviceProtection   n_unique=    3  ['No' 'No internet service' 'Yes']
11 TechSupport        n_unique=    3  ['No' 'No internet service' 'Yes']
12 StreamingTV        n_unique=    3  ['No' 'No internet service' 'Yes']
13 StreamingMovies    n_unique=    3  ['No' 'No internet service' 'Yes']
14 Contract           n_unique=    3  ['Month-to-month' 'One year' 'Two year']
15 PaperlessBilling   n_unique=    2  ['

In [ ]:
# Add-on columns: MultipleLines (6), OnlineSecurity ... StreamingMovies (8-13)
for i in [6, 8, 9, 10, 11, 12, 13]:
    X[:, i] = np.where(np.isin(X[:, i], ["No internet service", "No phone service"]), "No", X[:, i])

# Check: each column should now show only ['No' 'Yes']
for i in [6, 8, 9, 10, 11, 12, 13]:
    print(i, dataset.columns[1:-1][i], np.unique(X[:, i]))

6 MultipleLines ['No' 'Yes']
8 OnlineSecurity ['No' 'Yes']
9 OnlineBackup ['No' 'Yes']
10 DeviceProtection ['No' 'Yes']
11 TechSupport ['No' 'Yes']
12 StreamingTV ['No' 'Yes']
13 StreamingMovies ['No' 'Yes']


In [ ]:
for i in [6, 8, 9, 10, 11, 12, 13]:
    X[:, i] = le.fit_transform(X[:, i])   # No = 0, Yes = 1 (alphabetical)

In [ ]:
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [7, 14, 16])], remainder='passthrough')
X = np.array(ct.fit_transform(X))
X = X.astype(float)   # makes the whole array numeric for scaling/models


In [ ]:
cols = ct.get_feature_names_out()
pd.set_option("display.max_columns", None)   # show all 26 columns
pd.DataFrame(X, columns=cols).head(10)

,encoder__x7_DSL,encoder__x7_Fiber optic,encoder__x7_No,encoder__x14_Month-to-month,encoder__x14_One year,encoder__x14_Two year,encoder__x16_Bank transfer (automatic),encoder__x16_Credit card (automatic),encoder__x16_Electronic check,encoder__x16_Mailed check,remainder__x0,remainder__x1,remainder__x2,remainder__x3,remainder__x4,remainder__x5,remainder__x6,remainder__x8,remainder__x9,remainder__x10,remainder__x11,remainder__x12,remainder__x13,remainder__x15,remainder__x17,remainder__x18
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,29.85,29.85
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,34.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,56.95,1889.50
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,53.85,108.15
3,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,45.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,42.30,1840.75
4,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.70,151.65
5,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,8.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,99.65,820.50
6,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,22.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,89.10,1949.40
7,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,29.75,301.90
8,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,28.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,104.80,3046.05
9,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,62.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,56.15,3487.95


### **Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

In [ ]:
print(X_train)

[[0.00000e+00 0.00000e+00 1.00000e+00 ... 0.00000e+00 1.97000e+01
  2.58350e+02]
 [1.00000e+00 0.00000e+00 0.00000e+00 ... 0.00000e+00 7.39000e+01
  3.16055e+03]
 [1.00000e+00 0.00000e+00 0.00000e+00 ... 0.00000e+00 6.51500e+01
  4.68175e+03]
 ...
 [0.00000e+00 1.00000e+00 0.00000e+00 ... 1.00000e+00 1.05700e+02
  2.97950e+03]
 [1.00000e+00 0.00000e+00 0.00000e+00 ... 1.00000e+00 5.44000e+01
  1.14100e+02]
 [1.00000e+00 0.00000e+00 0.00000e+00 ... 0.00000e+00 6.82500e+01
  1.11485e+03]]


In [ ]:
print(X_test)

[[1.00000e+00 0.00000e+00 0.00000e+00 ... 1.00000e+00 7.98500e+01
  3.32075e+03]
 [0.00000e+00 1.00000e+00 0.00000e+00 ... 1.00000e+00 1.02400e+02
  6.47185e+03]
 [1.00000e+00 0.00000e+00 0.00000e+00 ... 1.00000e+00 4.50000e+01
  5.24350e+02]
 ...
 [1.00000e+00 0.00000e+00 0.00000e+00 ... 0.00000e+00 5.39500e+01
  3.88865e+03]
 [0.00000e+00 1.00000e+00 0.00000e+00 ... 1.00000e+00 8.57500e+01
  5.68845e+03]
 [0.00000e+00 1.00000e+00 0.00000e+00 ... 1.00000e+00 1.04200e+02
  1.74350e+03]]


In [ ]:
print(y_train)

[0 1 0 ... 1 1 0]


In [ ]:
print(y_test)

[0 0 0 ... 0 0 1]


In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = [14, 24, 25]   # tenure, MonthlyCharges, TotalCharges
sc = StandardScaler()

X_train[:, num_cols] = sc.fit_transform(X_train[:, num_cols])

# Standardisation of Values

In [ ]:
cols = ct.get_feature_names_out()
pd.set_option("display.max_columns", None)   # show all 26 columns
pd.DataFrame(X_train, columns=cols).head(10)

,encoder__x7_DSL,encoder__x7_Fiber optic,encoder__x7_No,encoder__x14_Month-to-month,encoder__x14_One year,encoder__x14_Two year,encoder__x16_Bank transfer (automatic),encoder__x16_Credit card (automatic),encoder__x16_Electronic check,encoder__x16_Mailed check,remainder__x0,remainder__x1,remainder__x2,remainder__x3,remainder__x4,remainder__x5,remainder__x6,remainder__x8,remainder__x9,remainder__x10,remainder__x11,remainder__x12,remainder__x13,remainder__x15,remainder__x17,remainder__x18
0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,-0.825884,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.497530,-0.890947
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.395961,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.302996,0.389693
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.577078,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.012320,1.060945
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.577078,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.686687,1.775397
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.092777,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.186726,-0.102671
5,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,-0.948068,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.007337,-0.712367
6,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.617806,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.141801,2.270166
7,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.174233,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.871058,0.153881
8,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,-1.070253,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.532214,-0.803245
9,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.599602,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.191709,0.455309


In [ ]:
X_test[:, num_cols] = sc.transform(X_test[:, num_cols])

# **Training and Evaluating the Logistic Regression Model**

In [74]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(X_train, y_train)

LogisticRegression(random_state=0)

In [75]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [0 0]
 [0 0]
 ...
 [0 0]
 [0 0]
 [1 1]]


In [77]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_auc_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, classifier.predict_proba(X_test)[:, 1]))

[[939 122]
 [144 204]]
0.8112136266855926
Precision: 0.6257668711656442
Recall   : 0.5862068965517241
ROC-AUC  : 0.8583355000162501


In [92]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_auc_score

y_prob = classifier.predict_proba(X_test)[:, 1]     # churn probabilities

'''# 1. Compare thresholds
print("Thresh  Recall  Precision  Accuracy")
for t in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]:
    yp = (y_prob >= t).astype(int)
    print(f"{t:5.2f}   {recall_score(y_test, yp):.3f}   {precision_score(y_test, yp):.3f}     {accuracy_score(y_test, yp):.3f}")
'''
# 2. Apply the chosen threshold
threshold = 0.3
y_pred_new = (y_prob >= threshold).astype(int)

print(f"\nThreshold = {threshold}")
print(confusion_matrix(y_test, y_pred_new))
print("Accuracy :", accuracy_score(y_test, y_pred_new))
print("Precision:", precision_score(y_test, y_pred_new))
print("Recall   :", recall_score(y_test, y_pred_new))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))   # unchanged by the threshold


Threshold = 0.3
[[794 267]
 [ 68 280]]
Accuracy : 0.7622427253371186
Precision: 0.5118829981718465
Recall   : 0.8045977011494253
ROC-AUC  : 0.8583355000162501


# **Training and Evaluating SVM Model**

In [78]:
from sklearn.svm import SVC
classifier2 = SVC(kernel = 'rbf', random_state = 0)
classifier2.fit(X_train, y_train)

SVC(random_state=0)

In [79]:
y_pred = classifier2.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [0 0]
 [0 0]
 ...
 [0 0]
 [0 0]
 [1 1]]


In [80]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_auc_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, classifier.predict_proba(X_test)[:, 1]))

[[954 107]
 [158 190]]
0.8119233498935415
Precision: 0.6397306397306397
Recall   : 0.5459770114942529
ROC-AUC  : 0.8583355000162501
